# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/muhammadfarhan2157-source/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*


Signal 1 : staleness (behind the refresh flags): checking whether older content really does perform worse.
Signal 2 : CTR vs. position (behind the CTR-fix logic): checking whether CTR really does drop as position worsens.
The rule, in plain words: flag a page if it's old-and-still-visible (stale) or has decent position but weak CTR. Score = average of the two flags. One reason code per page, one action label (refresh or monitor)

In [ ]:
path_march = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/*.parquet"

# Signal 1: staleness
staleness_check = con.sql(f"""
    SELECT
      CASE
        WHEN content_age_days < 90 THEN '0-90d'
        WHEN content_age_days < 180 THEN '90-180d'
        WHEN content_age_days < 365 THEN '180-365d'
        ELSE '365d+'
      END AS age_bucket,
      COUNT(*) AS n, AVG(clicks) AS avg_clicks, AVG(ctr) AS avg_ctr
    FROM '{path_march}' GROUP BY 1 ORDER BY 1
""").df()
print("Signal 1 — staleness. Verdict: [fill in: CONFIRMED/OPPOSITE/MIXED/FALSE]")
print(staleness_check)

# Signal 2: CTR vs position
ctr_position_check = con.sql(f"""
    SELECT
      CASE
        WHEN avg_position <= 3 THEN '1-3'
        WHEN avg_position <= 10 THEN '4-10'
        WHEN avg_position <= 20 THEN '11-20'
        ELSE '21+'
      END AS position_tier,
      COUNT(*) AS n, AVG(ctr) AS avg_ctr
    FROM '{path_march}' WHERE avg_position > 0 GROUP BY 1 ORDER BY 1
""").df()
print("Signal 2 — CTR vs position. Verdict: [fill in: CONFIRMED/OPPOSITE/MIXED/FALSE]")
print(ctr_position_check)


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
rule_df = con.sql(f"""
    SELECT
      content_hash_id, client_hash_id,
      content_age_days, avg_position, ctr, impressions, clicks,
      CASE WHEN content_age_days >= 180 AND impressions >= 500 THEN 1 ELSE 0 END AS stale_flag,
      CASE WHEN avg_position > 0 AND avg_position <= 20 AND ctr < 0.02 AND impressions >= 500 THEN 1 ELSE 0 END AS ctr_gap_flag
    FROM '{path_march}'
""").df()

rule_df["baseline_action_score"] = 0.5 * rule_df["stale_flag"] + 0.5 * rule_df["ctr_gap_flag"]

def reason_code(row):
    if row["stale_flag"] and row["ctr_gap_flag"]:
        return "stale_and_low_ctr"
    elif row["stale_flag"]:
        return "stale_visible_page"
    elif row["ctr_gap_flag"]:
        return "low_ctr_visible_page"
    return "no_flag"

rule_df["reason_code"] = rule_df.apply(reason_code, axis=1)
rule_df["action"] = rule_df["reason_code"].apply(lambda r: "refresh" if r != "no_flag" else "monitor")
rule_df = rule_df.sort_values("baseline_action_score", ascending=False)

import os
os.makedirs("../outputs", exist_ok=True)
rule_df.to_csv("../outputs/baseline_action_score.csv", index=False)
print("Wrote", len(rule_df), "rows to work/outputs/baseline_action_score.csv")


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
top20 = rule_df.head(20)[["content_hash_id", "baseline_action_score", "reason_code", "action",
                            "content_age_days", "avg_position", "ctr", "impressions"]]
top20


1. [content_hash_id] — action: refresh, reason: stale_and_low_ctr, confidence: high (both flags fire with real volume), would be wrong if: this is an intentionally static reference page where freshness doesn't matter.
2. [content_hash_id] — action: refresh, reason: stale_visible_page, confidence: medium (impressions just above the 500 floor), would be wrong if: those impressions came from a temporary spike, not steady demand.
(...continue for all 20 rows, one line each, genuinely tied to that row's actual numbers)

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

Weak picks: [name 2-3 rows from the top 20 where a flag fired right at the threshold — e.g. impressions barely over 500, or CTR just under the cutoff — flag these as needing human review, not confident calls.]

Leakage check: confirming no product flags or future windows leaked into this rule.

In [ ]:

used_columns = ["content_age_days", "impressions", "avg_position", "ctr"]
product_flags = ["health_score", "priority_score", "action_type", "needs_ctr_fix", "is_quick_win"]

print("Columns used in the rule:", used_columns)
print("Any product flags used?:", any(c in used_columns for c in product_flags))
print("Any month other than 2026-03 referenced in the rule build?: No — only path_march used")


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.